In [4]:
# ===============================
# 🚀 LoRA Merge + ValueHead + Test
# ===============================


# ✅ Imports
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from trl import AutoModelForCausalLMWithValueHead

# === Configurações ===
LORA_REPO = "augustocsc/Se124M500KInfPrompt_EOS"
BASE_MODEL = "gpt2"
OUTPUT_DIR = "./modelo_final_para_ppo"
MODEL_HUB = "augustocsc/Se124M500KInfPrompt_EOS_Merged"
# === Carregar o tokenizer correto ===
tokenizer = AutoTokenizer.from_pretrained(LORA_REPO)
tokenizer.pad_token = tokenizer.eos_token

# === Carregar modelo base e ajustar os embeddings ===
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
base_model.resize_token_embeddings(len(tokenizer))  # Corrige shape para 50258

# Load the PEFT model
peft_model = PeftModel.from_pretrained(base_model, LORA_REPO)

# === Merge das LoRA weights (corretamente) ===
merged_model = peft_model.merge_and_unload()

# === Adicionar Value Head ao modelo mergeado ===
model = AutoModelForCausalLMWithValueHead.from_pretrained(merged_model)

# === Salvar modelo final para PPO ===
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


('./modelo_final_para_ppo/tokenizer_config.json',
 './modelo_final_para_ppo/special_tokens_map.json',
 './modelo_final_para_ppo/vocab.json',
 './modelo_final_para_ppo/merges.txt',
 './modelo_final_para_ppo/added_tokens.json',
 './modelo_final_para_ppo/tokenizer.json')

In [5]:
model.push_to_hub(MODEL_HUB)
tokenizer.push_to_hub(MODEL_HUB)

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/augustocsc/Se124M500KInfPrompt_EOS_Merged/commit/175b8a2750f170839ce04cb3dab9b1740fc83e92', commit_message='Upload tokenizer', commit_description='', oid='175b8a2750f170839ce04cb3dab9b1740fc83e92', pr_url=None, repo_url=RepoUrl('https://huggingface.co/augustocsc/Se124M500KInfPrompt_EOS_Merged', endpoint='https://huggingface.co', repo_type='model', repo_id='augustocsc/Se124M500KInfPrompt_EOS_Merged'), pr_revision=None, pr_num=None)

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from trl import AutoModelForCausalLMWithValueHead
# 🔁 Recarregar o modelo já mergeado + value head
from trl import AutoModelForCausalLMWithValueHead
MODEL_HUB = "augustocsc/Se124M100KInfPrompt_EOS_Merged"
#load model
model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_HUB)
tokenizer = AutoTokenizer.from_pretrained(MODEL_HUB)

# 🔁 Prompt de teste
PROMPT = """
vars: x_1, x_2, x_3, x_4, x_5, x_6, x_7, x_8, x_9, x_10
oper: *, **, +, -, /
cons: C
expr:"""

device = model.pretrained_model.device  # 👈 modelo base dentro do wrapper
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(device)

# 🔮 Geração
gen_tokens = output = model.generate(
            input_ids=input_ids,
            max_new_tokens=50,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            
        )

# Mostrar resposta
response = tokenizer.decode(gen_tokens[0], skip_special_tokens=False)
print("🧪 Resposta do modelo:\n")
print(response)


Some weights of the model checkpoint at augustocsc/Se124M100KInfPrompt_EOS_Merged were not used when initializing GPT2LMHeadModel: ['v_head.summary.bias', 'v_head.summary.weight']
- This IS expected if you are initializing GPT2LMHeadModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2LMHeadModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. 

🧪 Resposta do modelo:


vars: x_1, x_2, x_3, x_4, x_5, x_6, x_7, x_8, x_9, x_10
oper: *, **, +, -, /
cons: C
expr: x_1 + x_2 + C*x_8 + C*x_5**C<|endoftext|>
